In [20]:
import pandas as pd
import numpy as np
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import sklearn
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OrdinalEncoder
import optuna

ModuleNotFoundError: No module named 'optuna'

In [3]:
df_train = pd.read_csv('donnees/train.csv')
df_test = pd.read_csv('donnees/test.csv')

df_train = df_train.fillna('Inconnu').copy()

#TARGET : 'sales_quantity' la vente 
y = df_train['sales_quantity'].copy()

#FEATURES
FEATURES = ['year', 'family_label', 'sub_department_label', 'department_label', 'univers_label',
    'product_nature_label', 'brand_label', 'product_technicity_level', 'color_name',
    'average_price', 'nb_store_max', 'nb_store_min']

X = df_train[FEATURES].copy()
print(X.info())


<class 'pandas.DataFrame'>
RangeIndex: 91927 entries, 0 to 91926
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   year                      91927 non-null  int64  
 1   family_label              91927 non-null  str    
 2   sub_department_label      91927 non-null  str    
 3   department_label          91927 non-null  str    
 4   univers_label             91927 non-null  str    
 5   product_nature_label      91927 non-null  str    
 6   brand_label               91927 non-null  str    
 7   product_technicity_level  91927 non-null  str    
 8   color_name                91927 non-null  str    
 9   average_price             91927 non-null  float64
 10  nb_store_max              91927 non-null  int64  
 11  nb_store_min              91927 non-null  int64  
dtypes: float64(1), int64(3), str(8)
memory usage: 8.4 MB
None


In [18]:


X_tr = (X[X['year'] < 2022]).copy()
X_te = (X[X['year'] == 2022]).copy()

#Encodage des 'str' en 'categorie'
cat_cols = X_tr.select_dtypes(
    include=["object", "string"]
).columns.tolist()

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)


X_tr[cat_cols] = encoder.fit_transform(X_tr[cat_cols])
X_te[cat_cols] = encoder.transform(X_te[cat_cols])

y_tr = (df_train[df_train['year'] < 2022]['sales_quantity']).copy()
y_te = (df_train[df_train['year'] == 2022]['sales_quantity']).copy()



model_lgb = lgb.LGBMRegressor(
    ojective = 'regression',
    learning_rate = 0.05,
    random_state = 0,
)
model_lgb.fit(X_tr_ca,y_tr)

model_cb = CatBoostRegressor(
    learning_rate = 0.05,
    random_state = 0,
    depth = 10
)
model_cb.fit(X_tr,y_tr)


print(f'{np.sqrt(mean_squared_error(y_te,model_lgb.predict(X_te))):0f}')
print(f'{np.sqrt(mean_squared_error(y_te,model_cb.predict(X_te))):0f}')

[LightGBM] [Warning] Unknown parameter: ojective
[LightGBM] [Warning] Unknown parameter: ojective
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000514 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1980
[LightGBM] [Info] Number of data points in the train set: 66620, number of used features: 12
[LightGBM] [Info] Start training from score 8037.381612
0:	learn: 23409.1868000	total: 26ms	remaining: 26s
1:	learn: 22975.7836609	total: 51.5ms	remaining: 25.7s
2:	learn: 22588.8017919	total: 75.3ms	remaining: 25s
3:	learn: 22216.8202602	total: 96.5ms	remaining: 24s
4:	learn: 21884.3134776	total: 118ms	remaining: 23.4s
5:	learn: 21545.1078857	total: 140ms	remaining: 23.1s
6:	learn: 21218.7390738	total: 160ms	remaining: 22.7s
7:	learn: 20921.3490479	total: 183ms	remaining: 22.7s
8:	learn: 20626.2019731	total: 205ms	remaining: 22.6s
9:	learn: 2

Pour le lgbm
Paramètres qui influencent le plus la performance
- n_estimators : nombre d'arbres construits.
- learning_rate : vitesse d'apprentissage ; une valeur faible nécessite généralement davantage d'arbres.
- num_leaves : complexité des arbres.
- max_depth : profondeur maximale des arbres.
- min_child_samples : nombre minimal d'observations dans une feuille.
- subsample : fraction des observations utilisées par arbre.
- colsample_bytree : fraction des variables utilisées par arbre.